In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Setup Cell

from scipy.stats import spearmanr
from datasets import load_dataset
from transformers import AutoTokenizer
import ctypes, json, random
from pathlib import Path
from collections import Counter
import numpy as np, torch
from tqdm import tqdm
import re
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import LoraConfig
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
from collections import defaultdict



try:  # NixOS: make libcuda visible
    ctypes.CDLL("/run/opengl-driver/lib/libcuda.so.1", mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass

SEED = 1
MODEL = "swiss-ai/Apertus-v1.5-8B"
VERSION = "mixed"            # "document" or "sentence" (prompt version from Cell 8)
DATA = next(p for p in (Path("synthetic"), Path("."), Path("..") / "synthetic")
            if (p / f"train_{VERSION}.jsonl").exists())
print("data from:", DATA.resolve())
OUT     = Path(f"out/apertus-8b-rsd-{VERSION}")
MAX_LEN = 12288                # tokens per example; longer ones are dropped (raise if memory allows)
QLORA   = True                  # 4-bit base model (fits 32 GB comfortably); False = bf16 LoRA

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")

In [ ]:
# Load train and valid data sets

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

ds = load_dataset("json", data_files={"train": str(DATA / f"train_{VERSION}.jsonl"),
                                      "valid": str(DATA / f"valid_{VERSION}.jsonl")})

def to_prompt_completion(ex):
    n = len(tok.apply_chat_template(ex["messages"], tokenize=True, return_dict=True)["input_ids"])

    return {"prompt": [ex["messages"][0]], "completion": [ex["messages"][1]], "n_tokens": n}

ds = ds.map(to_prompt_completion, remove_columns=["messages"], num_proc=8)
for split in ds:
    n = np.array(ds[split]["n_tokens"])
    print(f"{split}: {len(n):,} examples | tokens median {int(np.median(n)):,} p95 {int(np.quantile(n, .95)):,} "
          f"| dropped > {MAX_LEN}: {int((n > MAX_LEN).sum())} ({(n > MAX_LEN).mean():.1%})")
ds = ds.filter(lambda ex: ex["n_tokens"] <= MAX_LEN)
print(torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")

In [ ]:
# Show effects of difference share distrubution

def diff_share(ex):
    out = json.loads(ex["completion"][0]["content"])
    labs = np.array([s for k in ("sentence1", "sentence2") for _, s in out[k] if s != -1])
    return {"diff_share": float(np.mean(labs != 5)) if len(labs) else 0.0}

shares = np.array(ds["train"].map(diff_share, num_proc=8)["diff_share"])
print(f"now: {len(shares):,} docs | mean share of differences {shares.mean():.1%}\n")
u = np.random.default_rng(SEED).random(len(shares))
print("MIN_DIFF  KEEP_LOW   docs kept   share of differences")
for min_diff in (0.03, 0.05, 0.08):
    for keep_low in (0.2, 0.3, 0.5):
        keep = (shares >= min_diff) | (u < keep_low)
        print(f"  {min_diff:.2f}      {keep_low:.1f}      {keep.mean():6.1%}      {shares[keep].mean():6.1%}")

In [ ]:
# Rebalance the data set

MIN_DIFF, KEEP_LOW = 0.08, 0.3

def diff_share(ex):
    out = json.loads(ex["completion"][0]["content"])
    labs = np.array([s for k in ("sentence1", "sentence2") for _, s in out[k] if s != -1])
    return {"diff_share": float(np.mean(labs != 5)) if len(labs) else 0.0}

ds = ds.map(diff_share, num_proc=8)
rng = np.random.default_rng(SEED)
keep = [i for i, s in enumerate(ds["train"]["diff_share"]) if s >= MIN_DIFF or rng.random() < KEEP_LOW]
before = np.mean(ds["train"]["diff_share"])
ds["train"] = ds["train"].select(keep)
print(f"train: {len(keep):,} documents kept | mean share of differences {before:.1%} -> "
      f"{np.mean(ds['train']['diff_share']):.1%}")
ds = ds.remove_columns("diff_share")

In [ ]:
# Load model and LoRA config

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True) if QLORA else None
model = AutoModelForMultimodalLM.from_pretrained(MODEL, dtype=torch.bfloat16, quantization_config=bnb,
                                                 device_map={"": 0})          # sdpa where supported
model.config.use_cache = False

# all linear layers inside the text model's transformer blocks (not vision/audio, not lm_head)
lin_names = sorted({n.split(".")[-1] for n, m in model.named_modules()
                    if n.startswith("model.language_model.layers.") and "Linear" in type(m).__name__})
print("LoRA targets:", lin_names)
lora = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=r"model\.language_model\.layers\.\d+\..*\.(" + "|".join(map(re.escape, lin_names)) + ")")

print(f"GPU after loading: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

In [ ]:
# Use weighted SFT

PILOT = os.environ.get("PILOT", "on") == "on" # True: ~2000 short documents and 1 epoch else ->  False: full run
EPOCHS = 1 if PILOT else 2
ACCUM  = 16
W_COPY, W_SCORE, W_NON5 = 0.1, 1.0, 3.0      # copied words | score tokens | extra factor for scores 0-4

# find scored tokens
V = len(tok)
dec       = tok.batch_decode([[i] for i in range(V)])
starts_br = torch.tensor([d.lstrip().startswith("]") for d in dec])
has_br    = torch.tensor(["]" in d for d in dec])
ends_dash = torch.tensor([d.rstrip().endswith("-") for d in dec])
digit_of  = torch.full((V,), -1, dtype=torch.long)
for d in range(6):
    probe = json.dumps({"sentence1": [["a", d], ["b", d]], "sentence2": [["c", d], ["e", d]]})
    penc = tok(probe, return_offsets_mapping=True, add_special_tokens=False)
    for m in re.finditer(rf", {d}\]", probe):
        c = m.start() + 2
        i = next(k for k, (a, b) in enumerate(penc["offset_mapping"]) if a <= c < b)
        digit_of[penc["input_ids"][i]] = d
print("score tokens:", {tok.convert_ids_to_tokens(i): int(digit_of[i])
                        for i in (digit_of >= 0).nonzero().flatten().tolist()})

def token_weights(ids, labels):
    """Weight per target token: 0 outside the answer, W_COPY for copied text, W_SCORE (x W_NON5) for scores."""
    dev = ids.device
    nxt, prv = torch.roll(ids, -1, dims=-1), torch.roll(ids, 1, dims=-1)
    d = digit_of.to(dev)[ids]
    is_score = (d >= 0) & (has_br.to(dev)[ids] | starts_br.to(dev)[nxt]) & ~ends_dash.to(dev)[prv]
    w = torch.full(ids.shape, W_COPY, device=dev)
    w[is_score] = W_SCORE
    w[is_score & (d < 5)] = W_SCORE * W_NON5
    ans = labels != -100
    return w * ans, is_score & ans, d

class WeightedSFTTrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model_accepts_loss_kwargs = False            # Trainer divides by ACCUM itself

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        base = model.get_base_model() if hasattr(model, "get_base_model") else model


        h = base.model.language_model(input_ids=inputs["input_ids"], attention_mask=inputs.get("attention_mask"),
                                      use_cache=False).last_hidden_state








        tgt = inputs["input_ids"][:, 1:]
        w, _, _ = token_weights(tgt, inputs["labels"][:, 1:])
        sel = w > 0
        H, T, Wt = h[:, :-1][sel], tgt[sel], w[sel]

        def chunk_ce(Hc, Tc):                                                  # recomputed in backward
            return F.cross_entropy(base.lm_head(Hc).float(), Tc, reduction="none")

        ce = torch.cat([checkpoint(chunk_ce, H[i:i + 1024], T[i:i + 1024], use_reentrant=False)
                        for i in range(0, len(T), 1024)])
        loss = (ce * Wt).sum() / Wt.sum()
        return (loss, {"loss": loss}) if return_outputs else loss


# data

train_ds, eval_ds = ds["train"], ds["valid"]
if PILOT:                        # short documents only, so the pilot is fast
    train_ds = train_ds.filter(lambda ex: ex["n_tokens"] <= 4096).shuffle(seed=SEED).select(range(2000))
    eval_ds  = eval_ds.filter(lambda ex: ex["n_tokens"] <= 4096).select(range(40))
else:                            # smaller eval set: eval every 200 steps on long docs is slow
    eval_ds = eval_ds.shuffle(seed=SEED).select(range(min(60, len(eval_ds))))
train_ds, eval_ds = train_ds.remove_columns("n_tokens"), eval_ds.remove_columns("n_tokens")

steps = int(np.ceil(len(train_ds) / ACCUM) * EPOCHS)
every = 10 if PILOT else 200
cfg = SFTConfig(
    output_dir=str(OUT) + ("-weighted-pilot" if PILOT else "-weighted"),
    max_length=MAX_LEN,
    packing=False,
    completion_only_loss=True,           # prompt tokens get label -100
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=ACCUM,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.03 * steps)),
    optim="paged_adamw_8bit" if QLORA else "adamw_torch_fused",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=1 if PILOT else 10,
    eval_strategy="steps", eval_steps=every,
    save_strategy="steps", save_steps=every, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    prediction_loss_only=True,           # eval only needs the loss (compute_loss returns no logits)
    per_device_eval_batch_size=1,
    report_to="none",
    seed=SEED,
)
trainer = WeightedSFTTrainer(model=model, args=cfg, processing_class=tok, peft_config=lora,
                             train_dataset=train_ds, eval_dataset=eval_ds)

# text progress bar instead of the HTML one (papermill only logs text output)
from transformers.trainer_callback import ProgressCallback
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
trainer.add_callback(ProgressCallback)

# sanity check for gold answer

b = trainer.data_collator([trainer.train_dataset[0]])
_, is_score, d = token_weights(b["input_ids"][:, 1:], b["labels"][:, 1:])
gold = [s for k in ("sentence1", "sentence2") for _, s in json.loads(train_ds[0]["completion"][0]["content"])[k] if s != -1]
print(f"score tokens found {int(is_score.sum())} | gold scores {len(gold)} | "
      f"non-5 found {int((is_score & (d < 5)).sum())} | gold non-5 {sum(s < 5 for s in gold)}")
assert int(is_score.sum()) == len(gold), "score-token detection does not match"

print(f"{len(train_ds)} train / {len(eval_ds)} eval examples | {steps} optimizer steps")





from transformers.trainer_utils import get_last_checkpoint

last = get_last_checkpoint(cfg.output_dir) if Path(cfg.output_dir).is_dir() else None
if last:
    print("resuming from", last)
    trainer.train(resume_from_checkpoint=last)
else:
    print("no checkpoint found - starting fresh")
    trainer.train()




trainer.save_model(cfg.output_dir + "/final"); tok.save_pretrained(cfg.output_dir + "/final")

In [ ]:
# Select valid documents and check output


N_CHECK = 5
gen = trainer.model.eval(); gen.config.use_cache = True
decoder = json.JSONDecoder()

def pred_labels(items, n_gold, stats):
    """Labels from a predicted [[word, label], ...] list; malformed items count as 5 (= no difference)."""
    if not isinstance(items, list):
        stats["key missing / not a list"] += 1; items = []
    labels = []
    for it in items:
        if isinstance(it, list) and len(it) == 2 and isinstance(it[1], int) and -1 <= it[1] <= 5:
            labels.append(it[1])
        else:
            labels.append(5); stats["malformed items"] += 1
    stats["same length" if len(labels) == n_gold else "length differs"] += 1
    return labels[:n_gold] + [5] * max(0, n_gold - len(labels))   # pad/truncate like the official evaluation

ys, ps, stats = [], [], Counter()
for ex in random.Random(SEED).sample(list(eval_ds), N_CHECK):
    prompt = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(gen.device)
    gold_text = ex["completion"][0]["content"]
    gold = json.loads(gold_text)
    n_gold = len(tok(gold_text, add_special_tokens=False)["input_ids"])
    budget = n_gold + 256
    with torch.no_grad():
        out = gen.generate(**ids, max_new_tokens=budget, do_sample=False)
    new = out[0, ids["input_ids"].shape[1]:]
    text = tok.decode(new, skip_special_tokens=True).strip()

    try:
        pred, end = decoder.raw_decode(text)               # first complete JSON object
        stats["parseable" + (" (+ text after JSON)" if text[end:].strip() else "")] += 1
    except json.JSONDecodeError as e:
        stats["truncated (budget reached)" if len(new) >= budget else "broken JSON"] += 1
        print(f"--- not parseable: {e.msg} at char {e.pos} | generated {len(new)} tokens, gold {n_gold}")
        print("    ... " + text[max(0, e.pos - 150):e.pos + 100].replace("\n", "\\n"))
        continue
    if not isinstance(pred, dict):
        stats["JSON is not an object"] += 1; continue

    for key in ("sentence1", "sentence2"):
        g = gold[key]
        p = pred_labels(pred.get(key), len(g), stats)
        for (_, gs), s in zip(g, p):
            if gs != -1:
                ys.append(gs); ps.append(s)

print("\n", dict(stats))
if ys:
    ys, ps = np.array(ys), np.array(ps)
    print(f"Spearman vs. synthetic labels: {100 * spearmanr(ys, ps).statistic:.1f} | "
          f"predicted 5: {np.mean(ps == 5):.1%} (labels: {np.mean(ys == 5):.1%})")
if len(ys):
    pred_diff, gold_diff = ps != 5, ys != 5
    print(f"differences found (recall): {np.mean(pred_diff[gold_diff]):.1%} | "
          f"marked words that are real differences (precision): "
          f"{np.mean(gold_diff[pred_diff]) if pred_diff.any() else float('nan'):.1%}")

In [ ]:
# Print raw model output

IDX = 0                                                  # which validation example
ex = eval_ds[IDX]
gen = trainer.model.eval(); gen.config.use_cache = True

prompt = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(gen.device)
gold_text = ex["completion"][0]["content"]
budget = len(tok(gold_text, add_special_tokens=False)["input_ids"]) + 256
with torch.no_grad():
    out = gen.generate(**ids, max_new_tokens=budget, do_sample=False)
new = out[0, ids["input_ids"].shape[1]:]
raw = tok.decode(new, skip_special_tokens=False)          # as generated, incl. special tokens

user = ex["prompt"][0]["content"]
print("=== INPUT ===\n" + user[user.index("## Input to Annotate"):])
print(f"\n=== MODEL OUTPUT ({len(new)} tokens, budget {budget}) ===\n{raw}")
print(f"\n=== TARGET ===\n{gold_text}")

Path("debug").mkdir(exist_ok=True)
Path(f"debug/output_{IDX}.txt").write_text(raw, encoding="utf-8")
Path(f"debug/target_{IDX}.txt").write_text(gold_text, encoding="utf-8")
print(f"\nsaved to debug/output_{IDX}.txt and debug/target_{IDX}.txt")

In [ ]:
# Test on docs

N_DOCS          = 60          # None = all 504 dev pairs; start small for timing
MAX_EVAL_TOKENS = 24000       # longer pairs are skipped (and counted)

gen  = trainer.model.eval()
base = gen.get_base_model() if hasattr(gen, "get_base_model") else gen
W    = base.lm_head.weight
SCORE = re.compile(r", (-?\d)\]")

TEMPLATE = eval_ds[0]["prompt"][0]["content"]                       # the training prompt, few-shot part
PREFIX   = TEMPLATE[:TEMPLATE.index("## Input to Annotate")] + "## Input to Annotate\n\n"

def detok(tokens):
    """SwissGov stores space-separated tokens; make light running text like in training."""
    s = " ".join(tokens)
    s = re.sub(r" ([.,;:!?)\]%])", r"\1", s)
    return re.sub(r"([(\[]) ", r"\1", s)

def build(t1, t2):
    msg = [{"role": "user", "content": PREFIX + f"Sentence 1: {json.dumps([t1], ensure_ascii=False)}\n"
                                               f"Sentence 2: {json.dumps([t2], ensure_ascii=False)}\n\n"
                                               "Respond with the JSON object."}]
    return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def doc_probs(toks_a, labs_a, toks_b, labs_b):
    """P(score=0..5) for every item of both texts; all scores in the context = 5 (punctuation -1)."""
    items = lambda ts, ls: [[t, -1 if l == -1 else 5] for t, l in zip(ts, ls)]
    prompt = build(detok(toks_a), detok(toks_b))
    answer = json.dumps({"sentence1": items(toks_a, labs_a), "sentence2": items(toks_b, labs_b)}, ensure_ascii=False)
    enc  = tok(prompt + answer, return_offsets_mapping=True, add_special_tokens=False, return_tensors="pt")
    if enc["input_ids"].shape[1] > MAX_EVAL_TOKENS:
        return None
    ends = enc.pop("offset_mapping")[0][:, 1].numpy()
    slots = list(SCORE.finditer(answer))
    assert len(slots) == len(toks_a) + len(toks_b), "slot count mismatch"
    h = base.model(**enc.to(gen.device)).last_hidden_state[0]
    P = np.full((len(slots), 6), np.nan)
    groups = defaultdict(list)                                      # slots that share the same 6 token ids
    for j, m in enumerate(slots):
        if m.group(1) == "-1":
            continue
        i = int(np.searchsorted(ends, len(prompt) + m.start(1), side="right"))   # token holding the digit
        t = tok.convert_ids_to_tokens(int(enc["input_ids"][0, i]))
        ids = tuple(tok.convert_tokens_to_ids(t.replace("5", str(d))) for d in range(6))
        assert tok.unk_token_id not in ids, f"no score tokens for {t!r}"
        groups[ids].append((j, i - 1))
    for ids, js in groups.items():
        rows = torch.tensor([p for _, p in js], device=h.device)
        logits = (h[rows].float() @ W[list(ids)].float().T)
        P[[j for j, _ in js]] = torch.softmax(logits, -1).cpu().numpy()
    return P[:len(toks_a)], P[len(toks_a):]

rows = list(load_dataset("ZurichNLP/SwissGov-RSD", split="validation"))
if N_DOCS:
    rows = random.Random(SEED).sample(rows, N_DOCS)
gold, exp_d, arg_d, lang, skipped = [], [], [], [], 0
for r in tqdm(rows):
    ta, tb = r["text_a"].split(), r["text_b"].split()
    res = doc_probs(ta, r["labels_a"], tb, r["labels_b"])
    if res is None:
        skipped += 1; continue
    for P, labs in zip(res, (r["labels_a"], r["labels_b"])):
        m = np.array(labs) != -1
        gold += list(np.array(labs)[m]); lang += [r["subset"].split("_")[-1]] * int(m.sum())
        exp_d += list((P[m] @ (5 - np.arange(6))) / 5)              # expected difference 0..1
        arg_d += list((5 - P[m].argmax(1)) / 5)                     # what greedy decoding would write
    torch.cuda.empty_cache()

gold, exp_d, arg_d, lang = map(np.array, (gold, exp_d, arg_d, lang))
print(f"{len(rows) - skipped} docs scored | skipped (> {MAX_EVAL_TOKENS} tokens): {skipped}")
for name, pred in (("greedy-equivalent (argmax)", arg_d), ("score probabilities", exp_d)):
    per = " | ".join(f"en-{l} {100 * spearmanr(gold[lang == l], pred[lang == l]).statistic:.1f}"
                     for l in ("de", "fr", "it") if (lang == l).any())
    print(f"{name:<28} all {100 * spearmanr(gold, pred).statistic:.1f} | {per}")
print(f"argmax = 5 for {np.mean(arg_d == 0):.1%} of words (gold equivalent: {np.mean(gold == 0):.1%})")
print("reference: labeler on SwissGov dev = 31.0")

In [ ]:
# Diagnostics

SCORE = re.compile(r", (-?\d)\]")
gen = trainer.model.eval()

def score_probs(ex):
    """P(score = 0..5) per non-punctuation slot; all scores in the context set to 5 (no gold leakage)."""
    prompt = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
    gold_text = ex["completion"][0]["content"]
    gold = [m.group(1) for m in SCORE.finditer(gold_text)]
    ctx_text = SCORE.sub(lambda m: m.group(0) if m.group(1) == "-1" else ", 5]", gold_text)   # same length
    enc = tok(prompt + ctx_text, return_offsets_mapping=True, add_special_tokens=False, return_tensors="pt")
    offs = enc.pop("offset_mapping")[0].tolist()
    with torch.no_grad():
        logits = gen(**enc.to(gen.device)).logits[0].float()
    out = []
    for g, m in zip(gold, SCORE.finditer(ctx_text)):
        if g == "-1":
            continue
        c = len(prompt) + m.start(1)
        i = next(k for k, (a, b) in enumerate(offs) if a <= c < b)
        t = tok.convert_ids_to_tokens(int(enc["input_ids"][0, i]))
        ids = [tok.convert_tokens_to_ids(t.replace(m.group(1), str(d))) for d in range(6)]
        p = torch.softmax(logits[i - 1, ids], -1).cpu().numpy()
        out.append((int(g), p))
    return out

ys, exp_diff, p5 = [], [], []
for ex in tqdm(random.Random(SEED).sample(list(eval_ds), min(20, len(eval_ds)))):
    for gold, p in score_probs(ex):
        ys.append(gold); p5.append(p[5]); exp_diff.append(float(p @ (5 - np.arange(6)) / 5))
    torch.cuda.empty_cache()
ys, exp_diff = np.array(ys), np.array(exp_diff)
print(f"{len(ys):,} score slots | mean P(5) {np.mean(p5):.2f} | labels = 5: {np.mean(ys == 5):.1%}")
print(f"LEAK-FREE Spearman(expected difference, synthetic labels): {100 * spearmanr(-ys, exp_diff).statistic:.1f}")